<a href="https://colab.research.google.com/github/pranatixsharma/Masculine_defaults_Indian_youtube/blob/main/get_dfs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Cell 1 — Config + imports

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Transcripts_CSS"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Transcripts_CSS"

COMMUNITIES = [
    "transcripts_business",
    "transcripts_religion",
    "transcripts_comedy",
    "transcripts_lifestyle",
    "transcripts_tech",
    "transcripts_motivational",
    "transcripts_gaming",
    "transcripts_politics",
]

DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')
LATIN_RE = re.compile(r'[a-zA-Z]')

# thresholds — tune these once you see the distribution; YouTube videos
# run much shorter than the base paper's 10-min podcast cutoff
MIN_DURATION_SECONDS = 60      # was 10 minutes (600s) in base paper — adjust as needed
MIN_WORD_COUNT = 10

  # os.makedirs("./csv", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.makedirs("./csv", exist_ok=True)

##Cell 2 — Helper functions (Unicode-safe, no ASCII stripping)

In [ ]:
def get_unicode_text(dictionary):
    """
    Concatenate all segment text. Unlike the base paper's get_ascii_text,
    this keeps Devanagari characters intact — no .encode('ascii','ignore').
    """
    text = ""
    for seg in dictionary.get("segments", []):
        text += seg.get("text", "")
    return text.strip()

def get_duration(dictionary):
    """
    Approximate duration from segment timestamps, since there's no
    top-level duration field in the JSON (unlike Spotify metadata).
    """
    segments = dictionary.get("segments", [])
    if not segments:
        return 0.0
    return float(segments[-1].get("end", 0.0))

def classify_script(text, whisperx_lang, dev_thresh=0.9, latin_thresh=0.9,
                     mix_floor=0.05, other_dominant=0.5):
    """
    Returns (script_class, hindi_pct, english_pct) — percentages are
    share of Devanagari/Latin characters among all alphabetic characters
    (Devanagari + Latin + other-script). Rounded to 2 decimals.
    """
    if whisperx_lang == "nn":
        return "noise", 0.0, 0.0

    devanagari_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    latin_chars = sum(1 for c in text if c.isascii() and c.isalpha())
    other_chars = sum(1 for c in text if c.isalpha() and not c.isascii()
                       and not ('\u0900' <= c <= '\u097F'))

    total = devanagari_chars + latin_chars + other_chars
    if total == 0:
        return "unknown/no_text", 0.0, 0.0

    dev_pct = round(devanagari_chars / total * 100, 2)
    latin_pct = round(latin_chars / total * 100, 2)
    other_pct = other_chars / total

    if other_pct >= other_dominant:
        script_class = "other"
    elif dev_pct / 100 >= dev_thresh:
        script_class = "hindi"
    elif latin_pct / 100 >= latin_thresh:
        script_class = "english"
    elif dev_pct / 100 >= mix_floor and latin_pct / 100 >= mix_floor:
        script_class = "hindi-english (code-mixed)"
    else:
        script_class = "other"

    return script_class, dev_pct, latin_pct

def word_count(text):
    """
    Whitespace-based count works for both Devanagari and Latin script
    (Hindi uses spaces between words same as English), but strip
    punctuation-only tokens first so they don't inflate the count.
    """
    if not text:
        return 0
    tokens = [t for t in text.split() if any(c.isalnum() for c in t)]
    return len(tokens)

##Cell 3 — Walk all 8 communities and build the main df (slow — run once)

In [ ]:
records = []
error_files = []

for community in COMMUNITIES:
    folder_path = os.path.join(BASE_PATH, community)
    if not os.path.isdir(folder_path):
        print(f"WARNING: folder not found, skipping: {folder_path}")
        continue

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                full_path = os.path.join(root, file)
                try:
                    with open(full_path, encoding="utf-8") as f:
                        data = json.loads(f.read())

                    video_id = data.get("filename", file)
                    whisperx_lang = data.get("language", "unknown")
                    text = get_unicode_text(data)
                    duration = get_duration(data)
                    script_class, hindi_pct, english_pct = classify_script(text, whisperx_lang)
                    n_words = word_count(text)

                    record = {
                    "video_id": video_id,
                    "community": community,
                    "whisperx_language": whisperx_lang,
                    "script_class": script_class,
                    "hindi_pct": hindi_pct,
                    "english_pct": english_pct,
                    "transcript": text,                      # only in get_dfs.ipynb
                    "transcript_length": n_words,            # only in get_dfs.ipynb
                    "duration": duration,                    # only in get_dfs.ipynb
                    "file_path": full_path,
                    }
                    records.append(record)

                except Exception as e:
                    error_files.append((full_path, str(e)))

print(f"Total JSON files processed: {len(records)}")
print(f"Files with errors: {len(error_files)}")

df = pd.DataFrame(records)
print(df.shape)
df.head()

Total JSON files processed: 9461
Files with errors: 0
(9461, 10)


,video_id,community,whisperx_language,script_class,hindi_pct,english_pct,transcript,transcript_length,duration,file_path
0,CARachanaRanade__ayToSzjxzrE.mp3,transcripts_business,en,english,0.0,100.0,"Hey folks, CA Rachana Ranade here.Today's vide...",1723,599.958,/content/drive/MyDrive/Transcripts_CSS/transcr...
1,CARachanaRanade__azj4iFcL5aM.mp3,transcripts_business,en,english,0.0,100.0,"Hey, folks.CA Rachana Ranade here.And I welcom...",1383,600.018,/content/drive/MyDrive/Transcripts_CSS/transcr...
2,CARachanaRanade__bZRA5a9PJrY.mp3,transcripts_business,en,english,0.0,100.0,"Hey folks, CA Rachana Ranad here.Today's video...",1740,600.038,/content/drive/MyDrive/Transcripts_CSS/transcr...
3,CARachanaRanade__bfleN2D4zh4.mp3,transcripts_business,en,english,0.0,100.0,"Hey folks, CA Rachana Ranade here.You know, Q4...",1887,599.978,/content/drive/MyDrive/Transcripts_CSS/transcr...
4,CARachanaRanade__cAooRke0jKk.mp3,transcripts_business,en,english,0.0,100.0,"Hey folks, CA Rachana Ranade here, and I welco...",1781,599.978,/content/drive/MyDrive/Transcripts_CSS/transcr...


##Cell 4 — Filter to English / Hindi / code-mixed only

In [ ]:
KEEP_CLASSES = ["english", "hindi", "hindi-english (code-mixed)"]

before = len(df)
df = df[df["script_class"].isin(KEEP_CLASSES)].copy()
print(f"Dropped {before - len(df)} rows (other/unknown language) — kept {len(df)}")

Dropped 0 rows (other/unknown language) — kept 9461


##Cell 5 — Duration filter

In [ ]:
before = len(df)
too_short_df = df[df["duration"] < MIN_DURATION_SECONDS]
too_short_df.to_csv("./csv/too_short_df.csv", header=True, index=False)

df = df[df["duration"] >= MIN_DURATION_SECONDS]
print(f"Dropped {before - len(df)} rows under {MIN_DURATION_SECONDS}s duration — kept {len(df)}")

Dropped 23 rows under 60s duration — kept 9438


##Cell 6 — Word count filter (mirrors base paper's zero-word / low-word handling)

In [ ]:
zero_words_df = df[df["transcript_length"] == 0]
zero_words_df.to_csv("./csv/zero_words_df.csv", header=True, index=False)
df = df[df["transcript_length"] != 0]

less_than_min_words_df = df[df["transcript_length"] < MIN_WORD_COUNT]
less_than_min_words_df.to_csv("./csv/less_than_min_words_df.csv", header=True, index=False)
df = df[df["transcript_length"] >= MIN_WORD_COUNT]

print(f"Final row count after all filters: {len(df)}")

Final row count after all filters: 9430


##Cell 7 — Save final df

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/Transcripts_CSS/outputs"
#duration filter
too_short_df.to_csv(os.path.join(OUTPUT_DIR, "too_short_df.csv"), header=True, index=False)

#word count filter
zero_words_df.to_csv(os.path.join(OUTPUT_DIR, "zero_words_df.csv"), header=True, index=False)
less_than_min_words_df.to_csv(os.path.join(OUTPUT_DIR, "less_than_min_words_df.csv"), header=True, index=False)

#final df save
df.to_csv(os.path.join(OUTPUT_DIR, "df.csv"), header=True, index=False)
print(f"Saved {os.path.join(OUTPUT_DIR, 'df.csv')}")

Saved /content/drive/MyDrive/Transcripts_CSS/outputs/df.csv


In [ ]:
#parallel split for  faster computation

def split_dataframe(big_df, n_parts):
    indices = np.array(big_df.index)
    parts_indices = np.array_split(indices, n_parts)
    return [big_df.loc[idx] for idx in parts_indices]

big_df = pd.read_csv(os.path.join(OUTPUT_DIR, "df.csv"))

for n_parts, prefix in [(2, "df"), (4, "df-4")]:
    split_dfs = split_dataframe(big_df, n_parts)
    lens = []
    for i, part_df in enumerate(split_dfs):
        lens.append(len(part_df))
        part_df.to_csv(os.path.join(OUTPUT_DIR, f"{prefix}-{i}.csv"), header=True, index=False)
    assert sum(lens) == len(big_df)
    print(f"Split into {n_parts} parts: {lens}")

Split into 2 parts: [4715, 4715]
Split into 4 parts: [2358, 2358, 2357, 2357]
